In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.0 MB/s eta 0:00:00


In [3]:
import os
import cv2
import pandas as pd
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
train_csv = "/kaggle/input/datasets/lohithgudla/mldataset/train.csv"
meta_csv = "/kaggle/input/datasets/lohithgudla/mldataset/train_meta.csv"

df = pd.read_csv(train_csv)
meta = pd.read_csv(meta_csv)

print("Train rows:", len(df))
print("Meta rows:", len(meta))

Train rows: 67914
Meta rows: 15000


In [5]:
df = df[df["class_id"] != 14]

print("After removing No finding:", len(df))

After removing No finding: 36096


In [6]:
df = df.dropna(subset=["x_min","y_min","x_max","y_max"])

print("Valid annotations:", len(df))

Valid annotations: 36096


In [7]:
size_dict = {}

for _, row in meta.iterrows():
    size_dict[row["image_id"]] = (row["dim0"], row["dim1"])

In [8]:
base = "/kaggle/working/dataset"

img_dir = f"{base}/images/train"
lbl_dir = f"{base}/labels/train"

os.makedirs(img_dir, exist_ok=True)
os.makedirs(lbl_dir, exist_ok=True)

In [9]:
for img_id, group in tqdm(df.groupby("image_id")):

    h, w = size_dict[img_id]

    label_path = os.path.join(lbl_dir, img_id + ".txt")

    with open(label_path, "w") as f:

        for _, row in group.iterrows():

            x_min = row["x_min"]
            y_min = row["y_min"]
            x_max = row["x_max"]
            y_max = row["y_max"]

            xc = ((x_min + x_max) / 2) / w
            yc = ((y_min + y_max) / 2) / h

            bw = (x_max - x_min) / w
            bh = (y_max - y_min) / h

            class_id = int(row["class_id"])

            f.write(f"{class_id} {xc} {yc} {bw} {bh}\n")

100%|██████████| 4394/4394 [00:02<00:00, 1678.22it/s]


In [10]:
src_img_dir = "/kaggle/input/datasets/lohithgudla/mldataset/train"

for img in tqdm(os.listdir(src_img_dir)):
    
    src = os.path.join(src_img_dir, img)
    dst = os.path.join(img_dir, img)

    if not os.path.exists(dst):
        os.system(f"cp {src} {dst}")

100%|██████████| 15001/15001 [02:11<00:00, 114.20it/s]


In [11]:
images = set([os.path.splitext(f)[0] for f in os.listdir(img_dir)])
labels = set([os.path.splitext(f)[0] for f in os.listdir(lbl_dir)])

missing = images - labels

for img in missing:
    open(os.path.join(lbl_dir, img + ".txt"), "w").close()

print("Empty labels created:", len(missing))

Empty labels created: 10607


In [12]:
print("Images:", len(os.listdir(img_dir)))
print("Labels:", len(os.listdir(lbl_dir)))

Images: 15001
Labels: 15001


In [13]:
data_yaml = """
train: /kaggle/working/dataset/images/train
val: /kaggle/working/dataset/images/train

nc: 14

names:
  0: Aortic enlargement
  1: Atelectasis
  2: Calcification
  3: Cardiomegaly
  4: Consolidation
  5: ILD
  6: Infiltration
  7: Lung Opacity
  8: Nodule/Mass
  9: Other lesion
  10: Pleural effusion
  11: Pleural thickening
  12: Pneumothorax
  13: Pulmonary fibrosis
"""

with open("/kaggle/working/data.yaml","w") as f:
    f.write(data_yaml)

In [14]:
model = YOLO("yolo11s.pt")

model.train(
    data="/kaggle/working/data.yaml",
    epochs=40,
    imgsz=1024,
    batch=8
)

Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7daa50b3ae70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.0

In [15]:
model.predict(
    source="/kaggle/input/datasets/lohithgudla/mldataset/test",
    save=True,
    conf=0.25
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/3000 /kaggle/input/datasets/lohithgudla/mldataset/test/002a34c58c5b758217ed1f584ccbcfe9.png: 1024x1024 (no detections), 32.5ms
image 2/3000 /kaggle/input/datasets/lohithgudla/mldataset/test/004f33259ee4aef671c2b95d54e4be68.png: 1024x1024 (no detections), 32.4ms
image 3/3000 /kaggle/input/datasets/lohithgudla/mldataset/test/008bdde2af2462e86fd373a445d0f4cd.png: 1024x1024 2 Aortic enlargements, 1 Cardiomegaly, 32.5ms
image 4/3000 /kaggle/input/data

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'Aortic enlargement', 1: 'Atelectasis', 2: 'Calcification', 3: 'Cardiomegaly', 4: 'Consolidation', 5: 'ILD', 6: 'Infiltration', 7: 'Lung Opacity', 8: 'Nodule/Mass', 9: 'Other lesion', 10: 'Pleural effusion', 11: 'Pleural thickening', 12: 'Pneumothorax', 13: 'Pulmonary fibrosis'}
 obb: None
 orig_img: array([[[ 0,  0,  0],
         [ 0,  0,  0],
         [ 0,  0,  0],
         ...,
         [ 0,  0,  0],
         [ 0,  0,  0],
         [ 0,  0,  0]],
 
        [[ 0,  0,  0],
         [ 0,  0,  0],
         [ 0,  0,  0],
         ...,
         [ 0,  0,  0],
         [ 0,  0,  0],
         [ 0,  0,  0]],
 
        [[ 0,  0,  0],
         [ 0,  0,  0],
         [ 0,  0,  0],
         ...,
         [ 0,  0,  0],
         [ 0,  0,  0],
         [ 0,  0,  0]],
 
        ...,
 
        [[13, 13, 13],
         [14, 14, 14],
         [15, 15, 15]

In [16]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/runs/detect/train/weights/best.pt")

metrics = model.val(
    data="/kaggle/working/data.yaml",
    imgsz=1024
)

print(metrics)

Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,418,218 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2678.7±578.6 MB/s, size: 409.9 KB)
val: Scanning /kaggle/working/dataset/labels/train.cache... 15000 images, 10606 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 15000/15000 3.5Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 938/938 2.0it/s 7:48
                   all      15000      36096      0.455        0.4      0.399      0.213
    Aortic enlargement       3067       7162      0.673      0.636      0.688      0.443
           Atelectasis        186        279       0.36      0.219      0.238      0.105
         Calcification        452        960      0.334      0.326      0.287      0.136
          Cardiomegaly       2300       5427      0.749      0.573      0.661      0.456
         Consolidation     

In [17]:
import os
os.listdir("/kaggle/working/runs/detect/train")

['BoxPR_curve.png',
 'val_batch0_pred.jpg',
 'args.yaml',
 'results.png',
 'results.csv',
 'val_batch1_pred.jpg',
 'train_batch56250.jpg',
 'BoxR_curve.png',
 'train_batch0.jpg',
 'val_batch2_pred.jpg',
 'val_batch0_labels.jpg',
 'val_batch2_labels.jpg',
 'labels.jpg',
 'train_batch2.jpg',
 'confusion_matrix_normalized.png',
 'train_batch1.jpg',
 'train_batch56251.jpg',
 'weights',
 'BoxP_curve.png',
 'train_batch56252.jpg',
 'BoxF1_curve.png',
 'confusion_matrix.png',
 'val_batch1_labels.jpg']

In [18]:
import shutil

shutil.copy(
    "/kaggle/working/runs/detect/train/weights/best.pt",
    "/kaggle/working/cliniscan_detection_model.pt"
)

print("Model saved successfully")

Model saved successfully


In [19]:
from IPython.display import FileLink
FileLink("/kaggle/working/cliniscan_detection_model.pt")

/kaggle/working/cliniscan_detection_model.pt

In [20]:
import shutil

shutil.make_archive(
    "/kaggle/working/detection_results",
    'zip',
    "/kaggle/working/runs/detect/train"
)

'/kaggle/working/detection_results.zip'